In [1]:
using Healpix
using LinearAlgebra
using StaticArrays
using Base.Threads
using WignerD
using NPZ
using Plots
using Falcons
#using PyPlot
#using PyCall
using BenchmarkTools
#hp = pyimport("healpy")
#include("../src/EBC.jl")

In [2]:
mutable struct ConvolutionSky{I<:Int, MC<:Matrix{Complex{Float64}}}
    lmax::I
    alm::MC
    realization::I
end

mutable struct ConvolutionBeam{I<:Int, MC<:Matrix{Complex{Float64}}}
    lmax::I
    mmax::I
    blm::MC
end

#=
mutable struct ConvolutionCalculate{I<:Int, Bl<:Bool}
    nside_output::I
    lmax_calculate::I
    mmax_calculate::I
    HWP::Bl
end
=#

mutable struct ConvolutionCalculate{I<:Int, Bl<:Bool}
    nside_output::I
    lstart::I
    lstop::I
    mmax_calculate::I
    HWP::Bl
end



In [3]:
function ConvolutionSky(;
        lmax::Int = 3*128 - 1,
        alm::Matrix{ComplexF64} = fill(1.0 + 1.0im, 2, 2),
        realization::Int = 1
    )
    return ConvolutionSky(
        lmax,
        alm,
        realization
    )
end

function ConvolutionBeam(;
    lmax::Int = 3*128-1,
    mmax::Int = 2,
    blm::Matrix{ComplexF64} = fill(1.0 + 1.0im, 2, 2)
    )
    return ConvolutionBeam(lmax, mmax, blm)
end

function ConvolutionCalculate(;
    nside_output::Int = 128,
    lstart::Int = 0,
    lstop::Int = 3*nside_output - 1,
    mmax_calculate::Int = 2,
    HWP::Bool = false
)
    lstart <= lstop || throw(ArgumentError("lstart must be <= lstop (got lstart=$lstart, lstop=$lstop)"))
    mmax_calculate <= lstop || throw(ArgumentError("mmax_calculate must be <= lstop (got mmax_calculate=$mmax_calculate, lstop=$lstop)"))
    return ConvolutionCalculate(
        nside_output,
        lstart,
        lstop,
        mmax_calculate,
        HWP
    )
end


ConvolutionCalculate

In [4]:
#methods(ConvolutionSky)

In [4]:
cs = ConvolutionSky()
fieldnames(typeof(cs))

(:lmax, :alm, :realization)

In [5]:
cb = ConvolutionBeam()
fieldnames(typeof(cb))

(:lmax, :mmax, :blm)

In [6]:
cc = ConvolutionCalculate(nside_output = 128, lstart = 0, mmax_calculate = 2)
fieldnames(typeof(cc))

(:nside_output, :lstart, :lstop, :mmax_calculate, :HWP)

In [9]:
include("../../BeamToyModel/src/BeamToyModel.jl")

Main.BeamToyModel

In [10]:
nside_in = 128
lmax_in = 3nside_in-1
fwhm = deg2rad(1.0)

0.017453292519943295

In [11]:
blm_harmonic = BeamToyModel.gaussian_harmonic(lmax_in, fwhm, mmax = lmax_in)

(blmT = Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.28209479177387814 + 0.0im, 0.4885756718694027 + 0.0im, 0.6306791852123736 + 0.0im, 0.7461067059896269 + 0.0im, 0.8458196072043231 + 0.0im, 0.9348319547260208 + 0.0im, 1.0159345688951988 + 0.0im, 1.0908692242925033 + 0.0im, 1.1608087185364095 + 0.0im, 1.226586793157029 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], 383, 383, 767), blmE = Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], 383, 383, 767), blmB = Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0i

In [12]:
cb.lmax = lmax_in
cb.mmax = lmax_in
cb.blm = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm) 
;

In [13]:
nalm = Healpix.numberOfAlms(lmax_in, lmax_in)
almT = rand(ComplexF64, nalm)
almE = rand(ComplexF64, nalm)
almB = rand(ComplexF64, nalm)
cs.lmax = lmax_in
cs.alm = hcat(almT, almE, almB)
;

In [28]:
function lrange_idx(l::Int, lstart::Int)
    l >= lstart || throw(ArgumentError("l must be >= lstart"))
    offset = sum(2k + 1 for k in lstart:l-1; init=0)
    start = offset + 1
    stop  = start + (2l + 1) - 1
    return start, stop
end

function lmr_idx(;l::Int, m::Int, lstart::Int)
    l >= lstart || throw(ArgumentError("l must be >= lstart"))
    (-l <= m <= l) || throw(ArgumentError("m must be in [-l, l]"))
    offset = sum(2k + 1 for k in lstart:l-1; init=0)
    idx = offset + (m + l) + 1   # m=-l -> +1, m=l -> +2l+1
    return idx
end

function lmr_idx(; l::Int, m::Int, lstart::Int, mmax::Int)
    l ≥ lstart || throw(ArgumentError("l must be >= lstart"))

    mcap = min(l, mmax)
    (-mcap ≤ m ≤ mcap) || throw(ArgumentError("m must be in [-min(l,mmax), min(l,mmax)]"))

    offset = 0
    for k in lstart:l-1
        offset += 2*min(k, mmax) + 1
    end

    return offset + (m + mcap) + 1
end

function alm_idx(;l::Integer, m::Integer, lmax::Integer)
    return Int(m * (2 * lmax + 1 - m) // 2 + l)+1
end

function alm_idx(; l::Integer, m::Integer, lmax::Integer, mmax::Integer=lmax)
    (0 ≤ m ≤ mmax) || throw(ArgumentError("m must be in [0, mmax]"))
    (m ≤ l ≤ lmax) || throw(ArgumentError("l must satisfy m ≤ l ≤ lmax"))

    # offset for all previous m-blocks
    offset = m * (lmax + 1) - (m * (m - 1)) ÷ 2
    return Int(offset + (l - m) + 1)  # 1-based
end

function blm_lrange(cb, cc)
    n = sum(2*min(l, cc.mmax_calculate) + 1 for l in cc.lstart:cc.lstop)
    blm_calc = Matrix{ComplexF64}(undef, n, 3)
    for l in cc.lstart:cc.lstop
        m = 0
        idx_in = alm_idx(l=l, m=m, lmax=cb.lmax, mmax = cb.mmax)
        idx_out= lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.mmax_calculate)
        blm_calc[idx_out,1] = cb.blm[idx_in,1]
        blm_calc[idx_out,2] = -(cb.blm[idx_in,2] + 1im*cb.blm[idx_in,3]) # spin2 = -(E +iB)
        blm_calc[idx_out,3] = -(cb.blm[idx_in,2] - 1im*cb.blm[idx_in,3]) # spin-2 = -(E -iB)
        for m in 1:min(l, cc.mmax_calculate)
            phase = isodd(m) ? -1.0 : 1.0
            idx_in = alm_idx(l=l, m=m, lmax=cb.lmax, mmax = cb.mmax)
            idx_out_positive= lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.mmax_calculate)
            idx_out_negative= lmr_idx(l=l, m=-m, lstart=cc.lstart, mmax=cc.mmax_calculate)
            # m positive
            blm_calc[idx_out_positive,1] = cb.blm[idx_in,1]
            blm_calc[idx_out_positive,2] = -(cb.blm[idx_in,2] + 1im*cb.blm[idx_in,3]) # spin2 = -(E +iB)
            blm_calc[idx_out_positive,3] = -(cb.blm[idx_in,2] - 1im*cb.blm[idx_in,3]) # spin-2 = -(E -iB)
            # m negative
            blm_calc[idx_out_negative,1] = conj(cb.blm[idx_in,1])*phase # conj(spin0)*(-1)^m 
            blm_calc[idx_out_negative,2] = conj(blm_calc[idx_out_positive,3])*phase # conj(spin-2)*(-1)^m 
            blm_calc[idx_out_negative,3] = conj(blm_calc[idx_out_positive,2])*phase # conj(spin2)*(-1)^m 
        end
    end
    return blm_calc
end

blm_lrange (generic function with 1 method)

In [19]:
test = blm_lrange(cb, cc)

l = 0
l = 1
l = 2
l = 3
l = 4
l = 5
l = 6
l = 7
l = 8
l = 9
l = 10
l = 11
l = 12
l = 13
l = 14
l = 15
l = 16
l = 17
l = 18
l = 19
l = 20
l = 21
l = 22
l = 23
l = 24
l = 25
l = 26
l = 27
l = 28
l = 29
l = 30
l = 31
l = 32
l = 33
l = 34
l = 35
l = 36
l = 37
l = 38
l = 39
l = 40
l = 41
l = 42
l = 43
l = 44
l = 45
l = 46
l = 47
l = 48
l = 49
l = 50
l = 51
l = 52
l = 53
l = 54
l = 55
l = 56
l = 57
l = 58
l = 59
l = 60
l = 61
l = 62
l = 63
l = 64
l = 65
l = 66
l = 67
l = 68
l = 69
l = 70
l = 71
l = 72
l = 73
l = 74
l = 75
l = 76
l = 77
l = 78
l = 79
l = 80
l = 81
l = 82
l = 83
l = 84
l = 85
l = 86
l = 87
l = 88
l = 89
l = 90
l = 91
l = 92
l = 93
l = 94
l = 95
l = 96
l = 97
l = 98
l = 99
l = 100
l = 101
l = 102
l = 103
l = 104
l = 105
l = 106
l = 107
l = 108
l = 109
l = 110
l = 111
l = 112
l = 113
l = 114
l = 115
l = 116
l = 117
l = 118
l = 119
l = 120
l = 121
l = 122
l = 123
l = 124
l = 125
l = 126
l = 127
l = 128
l = 129
l = 130
l = 131
l = 132
l = 133
l = 134
l = 135
l = 136
l = 137
l = 13

1914×3 Matrix{ComplexF64}:
 0.282095+0.0im      -0.0-0.0im      -0.0-0.0im
     -0.0+0.0im       0.0-0.0im       0.0-0.0im
 0.488576+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0-0.0im   0.63061+0.0im      -0.0+0.0im
     -0.0+0.0im       0.0-0.0im       0.0-0.0im
 0.630679+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0+0.0im      -0.0-0.0im   0.63061-0.0im
      0.0-0.0im  0.746025+0.0im      -0.0+0.0im
     -0.0+0.0im       0.0-0.0im       0.0-0.0im
 0.746107+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0+0.0im      -0.0-0.0im      -0.0-0.0im
         ⋮                       
      0.0+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0+0.0im      -0.0-0.0im  0.143048-0.0im
      0.0-0.0im  0.140261+0.0im      -0.0+0.0im
     -0.0+0.0im       0.0-0.0im       0.0-0.0im
 0.140276+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0+0.0im      -0.0-0

In [25]:
n = sum(2*min(l, 3) + 1 for l in cc.lstart:cc.lstop)

2676

In [26]:
1+3+5+7+7*(cc.lstop-3)

2676

In [ ]:
n = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)


147456

In [99]:
alm_idx(l=4, m=3, lmax=4, mmax=3)

14

In [70]:
m = -2
phase = isodd(m) ? -1.0 : 1.0


1.0

In [ ]:
idx_start, idx_stop = lrange_idx(3, 2)

(6, 12)

In [58]:
lmr_idx(2, -2, 0)

5

In [46]:
maximum(cc.l_calculate)

383

In [ ]:
function lrange_idx(l::Int, lstart::Int)
    l >= lstart || throw(ArgumentError("l must be >= lstart"))
    offset = sum(2k + 1 for k in lstart:l-1)
    start = offset + 1
    stop  = start + (2l + 1) - 1
    return start, stop
end


In [43]:
for l in cc.l_calculate
    println("l = $l")
end

l = 0
l = 1
l = 2
l = 3
l = 4
l = 5
l = 6
l = 7
l = 8
l = 9
l = 10
l = 11
l = 12
l = 13
l = 14
l = 15
l = 16
l = 17
l = 18
l = 19
l = 20
l = 21
l = 22
l = 23
l = 24
l = 25
l = 26
l = 27
l = 28
l = 29
l = 30
l = 31
l = 32
l = 33
l = 34
l = 35
l = 36
l = 37
l = 38
l = 39
l = 40
l = 41
l = 42
l = 43
l = 44
l = 45
l = 46
l = 47
l = 48
l = 49
l = 50
l = 51
l = 52
l = 53
l = 54
l = 55
l = 56
l = 57
l = 58
l = 59
l = 60
l = 61
l = 62
l = 63
l = 64
l = 65
l = 66
l = 67
l = 68
l = 69
l = 70
l = 71
l = 72
l = 73
l = 74
l = 75
l = 76
l = 77
l = 78
l = 79
l = 80
l = 81
l = 82
l = 83
l = 84
l = 85
l = 86
l = 87
l = 88
l = 89
l = 90
l = 91
l = 92
l = 93
l = 94
l = 95
l = 96
l = 97
l = 98
l = 99
l = 100
l = 101
l = 102
l = 103
l = 104
l = 105
l = 106
l = 107
l = 108
l = 109
l = 110
l = 111
l = 112
l = 113
l = 114
l = 115
l = 116
l = 117
l = 118
l = 119
l = 120
l = 121
l = 122
l = 123
l = 124
l = 125
l = 126
l = 127
l = 128
l = 129
l = 130
l = 131
l = 132
l = 133
l = 134
l = 135
l = 136
l = 137
l = 13

In [62]:
hcat(blm_harmonic.blmT.alm,
     blm_harmonic.blmE.alm,
     blm_harmonic.blmB.alm) 

73920×3 Matrix{ComplexF64}:
 0.282095+0.0im  0.0+0.0im  0.0+0.0im
 0.488576+0.0im  0.0+0.0im  0.0+0.0im
 0.630679+0.0im  0.0+0.0im  0.0+0.0im
 0.746107+0.0im  0.0+0.0im  0.0+0.0im
  0.84582+0.0im  0.0+0.0im  0.0+0.0im
 0.934832+0.0im  0.0+0.0im  0.0+0.0im
  1.01593+0.0im  0.0+0.0im  0.0+0.0im
  1.09087+0.0im  0.0+0.0im  0.0+0.0im
  1.16081+0.0im  0.0+0.0im  0.0+0.0im
  1.22659+0.0im  0.0+0.0im  0.0+0.0im
  1.28882+0.0im  0.0+0.0im  0.0+0.0im
  1.34798+0.0im  0.0+0.0im  0.0+0.0im
  1.40444+0.0im  0.0+0.0im  0.0+0.0im
         ⋮                  
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0

In [59]:
[blm_harmonic.blmT.alm, blm_harmonic.blmE.alm, blm_harmonic.blmB.alm]

3-element Vector{Vector{ComplexF64}}:
 [0.28209479177387814 + 0.0im, 0.4885756718694027 + 0.0im, 0.6306791852123736 + 0.0im, 0.7461067059896269 + 0.0im, 0.8458196072043231 + 0.0im, 0.9348319547260208 + 0.0im, 1.0159345688951988 + 0.0im, 1.0908692242925033 + 0.0im, 1.1608087185364095 + 0.0im, 1.226586793157029 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im]
 [0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im]
 [0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im]

In [35]:
blm_harmonic.blmT.alm

73920-element Vector{ComplexF64}:
 0.28209479177387814 + 0.0im
  0.4885756718694027 + 0.0im
  0.6306791852123736 + 0.0im
  0.7461067059896269 + 0.0im
  0.8458196072043231 + 0.0im
  0.9348319547260208 + 0.0im
  1.0159345688951988 + 0.0im
  1.0908692242925033 + 0.0im
  1.1608087185364095 + 0.0im
   1.226586793157029 + 0.0im
   1.288820860640887 + 0.0im
  1.3479829400254342 + 0.0im
  1.4044432431790195 + 0.0im
                     ⋮
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im

In [47]:
mmax = lmax
nalm = Healpix.numberOfAlms(lmax, mmax)
blmT = Healpix.Alm(lmax, mmax, zeros(ComplexF64, nalm))

Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], 383, 383, 767)

In [52]:
blmT

Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], 383, 383, 767)

In [53]:
typeof(blmT)

Alm{ComplexF64, Vector{ComplexF64}}

In [4]:
function ConvolutionSky(;
    nside::Int = 128,
    lmax::Int = 3*nside - 1,
    alm::Matrix{ComplexF64} = fill(1.0 + 1.0im, 2, 2),
    realization::Int = 1
)
    return ConvolutionSky(nside, lmax, alm, realization)
end

ConvolutionSky

In [25]:
mutable struct ConvolutionSky{I<:Int, MC<:Matrix{Complex{Float64}}}
    nside::I
    lmax::I
    alm::MC
    realization::I
end

LoadError: invalid redefinition of constant Main.ConvolutionSky

In [5]:
nside = 16
cp = gen_ConvolutionParams_pc(nside = nside);
cp.beam_mmax = 10

10

In [6]:
lmax = nside*3-1
res = Resolution(nside)
beammmax = 10

10

In [7]:
# input beam
size = alm_idx(lmax,lmax,lmax)
blm_ = zeros(ComplexF64, 3, size)
blm = hp.blm_gauss(deg2rad(0.5), lmax = lmax, pol = true)
blm_[1,1:length(blm[1,:])] = blm[1,:]
blm_[2,1:length(blm[1,:])] = -blm[2,:]*sqrt(2)
blm_[3,1:length(blm[1,:])] = -blm[3,:]*sqrt(2);
@time blm_full = get_reorder_blm_optimized(blm_, lmax, beammmax);

LoadError: UndefVarError: `hp` not defined

In [8]:
alm = npzread("./inputs/alm=CMB=r0=nside$nside.npy")
@time alm_full = get_reorder_alm_optimized(alm, lmax);

  0.233391 seconds (245.43 k allocations: 16.078 MiB, 3.35% gc time, 98.94% compilation time)


In [9]:
data_path = "/data/n/n339/wigner_file/"

"/data/n/n339/wigner_file/"

In [10]:
@time initial_wignerd = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax]
@time initial_wignerd_ = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax]
@time eff_wignerD = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax];
@time eff_wignerD_ = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax];

  0.071945 seconds (29.82 k allocations: 4.259 MiB, 40.88% gc time, 57.19% compilation time)
  0.029705 seconds (21.82 k allocations: 3.724 MiB, 97.19% compilation time)
  0.028619 seconds (21.82 k allocations: 3.724 MiB, 96.99% compilation time)
  0.036557 seconds (21.82 k allocations: 3.724 MiB, 96.14% compilation time)


In [11]:
initial_wignerd[2]

3×3 Matrix{ComplexF64}:
 0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im

In [12]:
function initialwignerds_array_(cp, theta, path, initial_wignerd)
    count = 1
    for l in cp.l_range[1]:cp.l_range[2]
        initial_wignerd[count] = swignerd_calc(l, theta, path)
        count += 1
    end
   return initial_wignerd
end

initialwignerds_array_ (generic function with 1 method)

In [13]:
function effective_wignerD_onz_(cp, ψs, initial_wignerd_onz_)
    for m in -cp.lmax-2:cp.lmax+2
        initial_wignerd_onz_[cp.lmax+1+m] = mean(exp.(-1im*ψs*m))
    end
    return initial_wignerd_onz_
end

effective_wignerD_onz_ (generic function with 1 method)

In [14]:
npix = nside2npix(nside)
utv = unique_theta_val(cp);

In [15]:
initial_wignerd_onz = zeros(ComplexF64, 3, 2*cp.lmax+1)
conv_binned_map = zeros(3,npix)
result_d = zeros(ComplexF64,3)
@time for i in 1:length(utv)
    @show i
    start_pix, stop_pix = unique_theta_num(i, cp)
    initialwignerds_array(cp, utv[i], data_path, initial_wignerd);
    for pix_num in start_pix:stop_pix
        center_th, center_phi = pix2angRing(res, pix_num)
        calc_psi = rand(100)*2pi
        effective_wignerD_onz(cp, calc_psi[:], initial_wignerd_onz)
        for j in 1:3
            get_pc_total_effective_wignerD(cp, center_phi, initial_wignerd_onz[j,:], initial_wignerd, eff_wignerD);
            result_d[j] = eff_convolver_optimized(alm_full, blm_full, eff_wignerD, beammmax)
        end
        conv = binned_mapmaker(calc_psi, result_d)
        conv_binned_map[:,pix_num] .= conv
    end
end

i = 1


LoadError: SystemError: opening file "/data/n/n339/wigner_file/dmatrices=ell0.npy": No such file or directory

In [ ]:
alm_smooth = hp.smoothalm([alm[1,:],alm[2,:],alm[3,:]],fwhm = deg2rad(0.5))
in_map = hp.alm2map([alm_smooth[1],alm_smooth[2],alm_smooth[3]], nside = nside)

LoadError: UndefVarError: `alm` not defined